## Primo 1: Registro de vehículos con restricciones

**Referencia:** Medio 1 (claves primarias y restricciones)

**Contexto:** vas a simular una tabla de vehículos en memoria. Cada fila debe tener una matrícula única, una marca obligatoria y un kilometraje válido. Las restricciones representan las que aplicarías en una base de datos con `PRIMARY KEY`, `NOT NULL` y `CHECK`.

**Enunciado:**
1. Define la excepción `RestriccionError`, que hereda de `Exception` y no añade nada más.
2. Define la clase `Vehiculo` con `@dataclass(frozen=True)` y estos campos, en este orden: `matricula: str`, `marca: str` y `km: int`.
3. Define la clase `TablaVehiculos`. Su constructor guarda un diccionario vacío en `self._filas`, cuya clave será la matrícula.
4. Escribe `TablaVehiculos.insert(self, v)`. Comprueba las restricciones en este orden y lanza `RestriccionError` en el primer fallo:
   - Si la matrícula ya existe, el mensaje es `PRIMARY KEY duplicada: <matricula>`.
   - Si `marca` es una cadena vacía, el mensaje es `NOT NULL violado: marca`.
   - Si `km` es menor que 0, el mensaje es `CHECK violado: km <valor>`, con el valor formateado con `!r`.
   - Si todo es correcto, guarda la fila.
5. Escribe `TablaVehiculos.count(self)`, que devuelve el número de filas guardadas.
6. Crea la tabla y la lista `datos` con estos vehículos, en este orden: `Vehiculo("1111AAA", "Seat", 52000)`, `Vehiculo("2222BBB", "Ford", 0)`, `Vehiculo("1111AAA", "Opel", 10)`, `Vehiculo("3333CCC", "", 500)` y `Vehiculo("4444DDD", "Audi", -5)`.
7. Recorre `datos` e intenta insertar cada fila dentro de un `try/except RestriccionError`. Si se inserta, imprime `OK     matricula=<matricula>` (cinco espacios tras `OK`). Si falla, imprime `ERROR  <mensaje>` (dos espacios tras `ERROR`).
8. Imprime el recuento con `print("Filas:", tabla.count())`.
9. Imprime el resultado de comparar `Vehiculo("1111AAA", "Seat", 52000)` con `datos[0]` usando `==`.

**Recuerda:** `@dataclass(frozen=True)` genera, entre otros métodos, la comparación por igualdad y evita modificar las instancias después de crearlas. El orden de las comprobaciones determina qué error se informa cuando una fila incumple varias restricciones.

**Salida esperada:**
```text
OK     matricula=1111AAA
OK     matricula=2222BBB
ERROR  PRIMARY KEY duplicada: 1111AAA
ERROR  NOT NULL violado: marca
ERROR  CHECK violado: km -5
Filas: 2
True
```

In [ ]:
from dataclasses import dataclass

class RestrictionError(Exception):
    pass

@dataclass (frozen=True)
class Vehiculo:
    matricula: str
    marca: str
    km: int
    
class TablaVehiculos:
    def __init__(self):
        self._filas = {}
        
    def insert(self, v):
        if v.matricula in self._filas:
            raise RestrictionError(f"Matrícula duplicada: {v.matricula}")
        if not v.marca:
            raise RestrictionError(f"NOT NULL violado: marca")
        if v.km < 0:
            raise RestrictionError(f"CHECK violado: km {v.km!r}")
        self._filas[v.matricula] = v
    
    def count(self):
        return len(self._filas)

tabla = TablaVehiculos()
datos = [
    Vehiculo("1111AAA", "Seat", 52000),
    Vehiculo("2222BBB", "Ford", 0),
    Vehiculo("1111AAA", "Opel", 10),
    Vehiculo("3333CCC", "", 500), 
    Vehiculo("4444DDD", "Audi", -5),
]

for fila in datos:
    try:
        tabla.insert(fila)
        print(f"OK matricula={fila.matricula}")
    except RestrictionError as e:
        print(f"ERROR {e}")
print("Filas:", tabla.count())
print(Vehiculo("1111AAA", "Seat", 52000) == datos[0])




OK matricula=1111AAA
OK matricula=2222BBB
ERROR Matrícula duplicada: 1111AAA
ERROR NOT NULL violado: marca
ERROR CHECK violado: hm -5
Filas: 2
True


## Primo 2: Consumo por estación (`GROUP BY`)

**Referencia:** Medio 2 (agrupación y agregaciones)

**Contexto:** tienes lecturas de consumo eléctrico registradas por estación. Vas a agruparlas y calcular cuántas lecturas hay y cuánto se ha consumido en cada estación, como harías con `GROUP BY`, `COUNT` y `SUM` en una consulta SQL.

**Enunciado:**
1. Usa la lista `lecturas`, formada por tuplas `(id, estacion, consumo_wh)`, con el consumo entero en vatios-hora y estos valores en este orden: `(1, "Norte", 300)`, `(2, "Sur", 500)`, `(3, "Norte", 700)`, `(4, "Este", 1000)`, `(5, "Sur", 500)`, `(6, "Norte", 0)` y `(7, "Oeste", 1500)`.
2. Importa `defaultdict` de `collections`.
3. Escribe `agrupar_por(filas, clave)`. Recibe una lista y una función `clave`, y devuelve un `defaultdict(list)` que asocia cada valor de `clave(fila)` con la lista de filas de ese grupo, respetando el orden de entrada.
4. Escribe `resumir(filas)`. Agrupa por la estación (posición 1 de la tupla) y devuelve una lista de tuplas `(estacion, cantidad, suma_de_consumo)`. Ordena la lista por suma descendente y, en caso de empate, por nombre de estación ascendente.
5. Imprime una línea por grupo con este formato exacto: `f"{estacion:<6}| COUNT={n} | SUM={total} wh"`. El nombre se rellena con espacios a la derecha hasta 6 caracteres.
6. Al final, imprime `resumir([])`.

**Recuerda:** el criterio secundario de ordenación hace que los grupos con la misma suma aparezcan en orden alfabético. El especificador `<6` alinea el nombre a la izquierda dentro de un campo de seis caracteres.

**Salida esperada:**
```text
Oeste | COUNT=1 | SUM=1500 wh
Este  | COUNT=1 | SUM=1000 wh
Norte | COUNT=3 | SUM=1000 wh
Sur   | COUNT=2 | SUM=1000 wh
[]
```

In [8]:
from collections import defaultdict

lecturas = [
    (1, "Norte", 300),
    (2, "Sur", 500),
    (3, "Norte", 700),
    (4, "Este", 1000),
    (5, "Sur", 500),
    (6, "Norte", 0),
    (7, "Oeste", 1500),
]

def agrupar_por(filas, clave):
    grupos = defaultdict(list)
    for f in filas:
        grupos[clave(f)].append(f)
    return grupos

def resumir(filas):
    grupos = agrupar_por(filas, lambda f: f[1])
    return sorted(
        ((estacion, len(g), sum(f[2] for f in g)) for estacion, g in grupos.items()),
        key=lambda t: (-t[2], t[0]),
    )

for estacion, n, total in resumir(lecturas):
    print(f"{estacion:<6} | COUNT={n} | SUM={total} wh")
    
print(resumir([]))

Oeste  | COUNT=1 | SUM=1500 wh
Este   | COUNT=1 | SUM=1000 wh
Norte  | COUNT=3 | SUM=1000 wh
Sur    | COUNT=2 | SUM=1000 wh
[]


## Primo 3: Libros y autores (`JOIN`)

**Referencia:** Medio 3 (uniones entre tablas)

**Contexto:** una biblioteca guarda libros y autores en tablas distintas. Puedes relacionarlas con el identificador del autor, igual que una clave foránea apunta a una clave primaria. Simularás las operaciones `INNER JOIN` y `LEFT JOIN` usando listas de diccionarios.

**Enunciado:**
1. Define `libros`, una lista de diccionarios con las claves `id`, `titulo` y `autor_id`, en este orden: `{"id": 1, "titulo": "Quijote", "autor_id": 1}`, `{"id": 2, "titulo": "Ficciones", "autor_id": 2}`, `{"id": 3, "titulo": "Anonimo", "autor_id": None}` y `{"id": 4, "titulo": "Emma", "autor_id": 77}`.
2. Define `autores`, con las claves `id` y `nombre`, en este orden: `{"id": 1, "nombre": "Cervantes"}`, `{"id": 2, "nombre": "Borges"}` y `{"id": 3, "nombre": "Austen"}`.
3. Escribe `inner_join(izq, der, col_izq, col_der="id")`. Primero construye un diccionario índice del lado derecho, usando `col_der` como clave. Devuelve una lista de tuplas `(fila_izq, fila_der)` solo para las filas de la izquierda cuyo valor de `col_izq` exista en el índice, conservando el orden de la izquierda.
4. Escribe `left_join(izq, der, col_izq, col_der="id")`, usando el mismo tipo de índice. Devuelve una tupla por cada fila de la izquierda; si no hay pareja, el segundo elemento debe ser `None`.
5. Imprime `INNER JOIN` y, debajo, una línea por pareja con el formato `<titulo> -> <nombre del autor>`.
6. Imprime `LEFT JOIN` y, debajo, una línea por fila de la izquierda con el mismo formato. Cuando no hay autor, escribe `NULL`.
7. Imprime el resultado de `inner_join([], autores, "autor_id")`.

**Recuerda:** en un `INNER JOIN` solo aparecen las filas que tienen coincidencia en ambas colecciones. En un `LEFT JOIN` se conservan todas las filas de la izquierda; cuando no hay coincidencia, el resultado asociado es `None`.

**Salida esperada:**
```text
INNER JOIN
Quijote -> Cervantes
Ficciones -> Borges
LEFT JOIN
Quijote -> Cervantes
Ficciones -> Borges
Anonimo -> NULL
Emma -> NULL
[]
```

In [1]:
libros = [
    {"id": 1, "titulo": "Quijote", "autor_id": 1}, 
    {"id": 2, "titulo": "Ficciones", "autor_id": 2}, 
    {"id": 3, "titulo": "Anonimo", "autor_id": None},
    {"id": 4, "titulo": "Emma", "autor_id": 77},
]

autores = [
    {"id": 1, "nombre": "Cervantes"}, 
    {"id": 2, "nombre": "Borges"},
    {"id": 3, "nombre": "Austen"},
]

def inner_join(izq, der, col_izq, col_der="id"):
    indice = {dict[col_der]: dict for dict in der}
    return [(i, indice[i[col_izq]]) for i in izq if i[col_izq] in indice] 

def left_join(izq, der, col_izq, col_der="id"): 
    indice = {d[col_der]: d for d in der} 
    return [(i, indice.get(i[col_izq])) for i in izq] 

print("INNER JOIN") 
for e, d in inner_join(libros, autores , "autor_id"): 
    print(f"{e['titulo']} -> {d['nombre']}") 
print("LEFT JOIN") 
for e, d in left_join(libros, autores, "autor_id"): 
    print(f"{e['titulo']} -> {d['nombre'] if d else 'NULL'}") 
print(inner_join([], autores, "autor_id")) 


INNER JOIN
Quijote -> Cervantes
Ficciones -> Borges
LEFT JOIN
Quijote -> Cervantes
Ficciones -> Borges
Anonimo -> NULL
Emma -> NULL
[]


## Primo 4: Jugadores en tabla genérica

**Referencia:** Medio 4 (tablas y consultas genéricas)

**Contexto:** vas a implementar en memoria una tabla genérica con operaciones equivalentes a `INSERT` y `SELECT`. La función de clave primaria identifica cada fila; los predicados y criterios de ordenación representan las cláusulas `WHERE` y `ORDER BY`.

**Enunciado:**
1. Importa `Callable`, `Generic`, `Hashable` y `TypeVar` de `typing`, y declara `T = TypeVar("T")`.
2. Define la clase `Tabla(Generic[T])`. Su constructor recibe `pk: Callable[[T], Hashable]`, una función que extrae la clave primaria de una fila. Guarda `pk` y un diccionario vacío en `self._filas`.
3. Escribe `insert(self, fila)`. Calcula la clave con `pk`. Si ya existe, lanza `KeyError` con el mensaje `Codigo duplicado: <clave>`; si no, guarda la fila.
4. Escribe `select(self, where=<lambda que devuelve True>, order_by=None, desc=False)`. Filtra las filas con `where` y, si `order_by` no es `None`, ordena la lista resultante con ese criterio como clave y `reverse=desc`. Devuelve la lista.
5. Crea `tabla: Tabla[tuple] = Tabla(pk=lambda f: f[0])`. Las filas son tuplas `(codigo, nombre, puntos)`. Inserta, en este orden: `(10, "Zoe", 80)`, `(11, "Biel", 20)`, `(12, "Nuria", 150)` y `(13, "Hugo", 45)`.
6. Intenta insertar `(11, "Otro", 1)` dentro de un `try/except KeyError as e`. Si falla, imprime `Error:` y la excepción con `print("Error:", e)`.
7. Imprime `WHERE puntos < 100 ORDER BY puntos ASC` y, debajo, cada tupla del `select` con puntos menores que 100, ordenadas por puntos ascendente, una por línea.
8. Imprime `WHERE puntos < 0` y, debajo, el resultado de un `select` con puntos menores que 0.

**Recuerda:** `Generic[T]` permite parametrizar la tabla según el tipo de fila. `where` selecciona qué filas incluir y `order_by` determina la clave de ordenación; el parámetro `desc` permite invertirla.

**Salida esperada:**
```text
Error: 'Codigo duplicado: 11'
WHERE puntos < 100 ORDER BY puntos ASC
(11, 'Biel', 20)
(13, 'Hugo', 45)
(10, 'Zoe', 80)
WHERE puntos < 0
[]
```

## Primo 5: Cursor perezoso de canciones

**Referencia:** Medio 5 (cursores y paginación)

**Contexto:** un cursor permite leer filas bajo demanda. Combinado con la paginación mediante `LIMIT` y `OFFSET`, ayuda a procesar solo el tramo necesario sin cargar toda la información en memoria.

**Enunciado:**
1. Importa `islice` de `itertools`.
2. Escribe el generador `leer_filas(filas)`. Para cada fila imprime `[cargando <id>]`, con dos espacios al principio, y después devuelve la fila con `yield`.
3. Escribe `limit_offset(it, limit, offset=0)`, que devuelve una lista con `islice(it, offset, offset + limit)`.
4. Crea `filas = [{"id": i, "titulo": f"Tema{i}"} for i in range(1, 7)]`, es decir, seis canciones con títulos `Tema1` a `Tema6`.
5. Imprime `LIMIT 3 OFFSET 2`. Llama a `limit_offset` con un cursor nuevo, `limit=3` y `offset=2`. Imprime la lista de títulos de la página resultante.
6. Imprime `LIMIT 2 OFFSET 20`. Llama a `limit_offset` con otro cursor nuevo, `limit=2` y `offset=20`, e imprime la lista resultante.
7. Imprime `Cursor perezoso`. Crea un cursor `c = leer_filas(filas)` y llama dos veces, en líneas separadas, a `print(next(c)["titulo"])`.

**Recuerda:** `islice(it, inicio, fin)` toma elementos desde `inicio` hasta `fin - 1`. Cada cursor es independiente, así que cada paginación empieza a leer desde la primera fila de su propio iterador.

**Salida esperada:**
```text
LIMIT 3 OFFSET 2
  [cargando 1]
  [cargando 2]
  [cargando 3]
  [cargando 4]
  [cargando 5]
['Tema3', 'Tema4', 'Tema5']

LIMIT 2 OFFSET 20
  [cargando 1]
  [cargando 2]
  [cargando 3]
  [cargando 4]
  [cargando 5]
  [cargando 6]
[]

Cursor perezoso
  [cargando 1]
Tema1
  [cargando 2]
Tema2
```